In [ ]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")


# ============================================================
# SMOKE_TEST: validate pipeline before committing to real run
# (per PLAN.md discipline rule #6 — added 2026-05-19)
# When True: 20 records, max_steps=3, ~5-10 min runtime
# When False: full dgxchen training, ~80-90 min runtime
# ============================================================
SMOKE_TEST = False  # REAL RUN: full 10,311 rich-CoT records -> ~323 steps @ ~94-98s = ~8.5h on RTX (batch1/accum32 = eff batch 32, v13-proven config)

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/dgxchen/trained-adapter"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


In [ ]:
# === GPU GUARD ===
import torch
_v = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0.0
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'} | VRAM {_v:.1f} GB")
assert _v >= 70, f"VRAM {_v:.1f}GB too small for 30B bf16 scoring. Pick RTX Pro 6000 (96GB), Run All."
print("GPU OK for logprob scoring.")


In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    # GPU FAIL-FAST (eliminates the "ran on the wrong GPU" waste from v16): bf16 30B needs ~60 GB just
    # to load + training overhead, so only the RTX Pro 6000 (96 GB) works. Abort BEFORE the expensive
    # model load if a smaller accelerator was selected in the UI.
    _vram = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1e9
    _names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print(f"GPU(s): {_names} | total VRAM {_vram:.0f} GB")
    if _vram < 70:
        raise RuntimeError(f"Total VRAM {_vram:.0f} GB too small for bf16 30B. Select RTX Pro 6000 (96 GB) "
                           f"under Settings -> Accelerator, then Save & Run All.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
# === LOGPROB SCORING: base-model teacher-forced loss per trace (Plan #1, part a) ===
# One forward pass per trace (NO generation). High mean/max loss = base finds the trace "unlearnable" -> drop.
import torch, torch.nn.functional as F, json, glob, time
import pandas as pd

model.eval()
_c = glob.glob('/kaggle/input/**/v18-corpus.jsonl', recursive=True)
assert _c, "v18-corpus.jsonl not found - attach gideonnazarian/nemotron-v18-corpus"
records = [json.loads(l) for l in open(_c[0]) if l.strip()]
print(f"scoring {len(records)} traces from {_c[0]}", flush=True)

def score(msgs):
    full   = tokenizer.apply_chat_template(msgs,      tokenize=True, add_generation_prompt=False, enable_thinking=True)
    prompt = tokenizer.apply_chat_template(msgs[:1],  tokenize=True, add_generation_prompt=True,  enable_thinking=True)
    full = full[:MAX_SEQ_LEN]
    n_p = min(len(prompt), len(full))
    if n_p >= len(full):            # assistant fully truncated
        return None
    ids = torch.tensor([full], device=model.device)
    with torch.no_grad():
        logits = model(ids).logits[0]                                  # [L, V]
        loss = F.cross_entropy(logits[:-1].float(), ids[0,1:], reduction='none')  # [L-1] = -logprob/token
    asst = loss[n_p-1:]            # assistant-token losses only
    return float(asst.mean()), float(asst.max()), int(asst.numel())   # mean_loss, worst-token loss, n

rows=[]; t0=time.time(); sane=False
for i, rec in enumerate(records):
    try:
        out = score(rec['messages'])
    except Exception as e:
        print(f"  [skip {i}] {type(e).__name__}: {e}", flush=True); continue
    if out is None: continue
    ml, xl, n = out
    rows.append({'idx': i, 'type': rec.get('type','other'), 'n_asst': n,
                 'mean_loss': round(ml,5), 'max_loss': round(xl,5)})
    # --- SANITY GATE: after the first 5 valid scores, assert losses are plausible (catches tokenization bugs early) ---
    if (not sane) and len(rows)==5:
        ms=[r['mean_loss'] for r in rows]
        print(f"[SANITY] first 5 mean_loss = {ms}", flush=True)
        assert all(0.0 < x < 50.0 for x in ms), f"SANITY FAIL: implausible losses {ms} -> check tokenization/mask. STOP."
        print("[SANITY] OK - losses positive + reasonable; proceeding to full scoring.", flush=True); sane=True
    if (i+1)%500==0:
        print(f"  {i+1}/{len(records)} scored ({(time.time()-t0)/60:.1f}m, {len(rows)} kept)", flush=True)

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/base_loss.csv', index=False)
print(f"\nDONE: {len(df)} traces in {(time.time()-t0)/60:.1f}m -> /kaggle/working/base_loss.csv", flush=True)
print("\n=== mean_loss distribution (overall) ===")
print(df['mean_loss'].describe())
print("\n=== mean_loss by category (THE filter signal: high = unlearnable, drop candidates) ===")
print(df.groupby('type')['mean_loss'].agg(['count','mean','median','max']).round(3).sort_values('mean'))
